## Reranking

In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np


## load libraries
import os
from dotenv import load_dotenv
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# LangChain core imports
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import (
    RunnablePassthrough, 
 
)
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

# LangChain specific imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# Load environment variables
load_dotenv()

d:\course 2\RAG udemy\RAGUdemy\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [3]:
# Load documents
loader = TextLoader("langchain_sample.txt")
documents = loader.load()

# Split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 300, chunk_overlap = 50)
chunks = text_splitter.split_documents(documents)
chunks

[Document(metadata={'source': 'langchain_sample.txt'}, page_content='LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.'),
 Document(metadata={'source': 'langchain_sample.txt'}, page_content='LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.'),
 Document(metadata={'source': 'langchain_sample.txt'}, page_content='Retrieval-Augmented Generation (RAG) is a powerful technique where external knowledge is retrieved and passed into the prompt to ground LLM responses. LangChain makes it easy to implement RAG using vector databases like FAISS, Chroma, and Pinecone.'),
 Document(metadata={

In [4]:
#query
query = "How can I use Langchain to build an application that can answer questions based on a document?"

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
# Create FAISS vector store
vector_store = FAISS.from_documents(chunks, embeddings)
# Retrieve relevant documents
retriever = vector_store.as_retriever(search_kwargs={"k": 5})

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 298.32it/s]


In [7]:
#sparse retrieval
from langchain_community.retrievers import BM25Retriever
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 5

# Combine dense and sparse retrievers
from langchain_classic.retrievers import EnsembleRetriever
ensemble_retriever = EnsembleRetriever(retrievers=[retriever, bm25_retriever], weights=[0.7, 0.5])

In [10]:
ensemble_retriever.invoke(query)

[Document(id='36550579-0188-4a69-bb50-e2e148c6dd45', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.'),
 Document(id='d5c8c1ca-c61e-43c8-b1fc-bf4193c189d3', metadata={'source': 'langchain_sample.txt'}, page_content='Agents in LangChain are chains that use LLMs to decide which tools to use and in what order. This makes them suitable for multi-step tasks like question answering with search and code execution.'),
 Document(id='d2024728-1dbe-49ab-8236-4ee6aa9b2a4a', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components

In [20]:
from langchain_classic.llms import Ollama
llm = Ollama(model="phi3")

In [21]:
# Prompt Template
prompt = PromptTemplate.from_template("""
You are a helpful assistant. Your task is to rank the following documents from most to least relevant to the user's question.

User Question: "{question}"

Documents:
{documents}

Instructions:
- Think about the relevance of each document to the user's question.
- Return a list of document indices in ranked order, starting from the most relevant.

Output format: comma-separated document indices (e.g., 2,1,3,0,...)
""")

In [22]:
retrieved_docs=ensemble_retriever.invoke(query)
retrieved_docs

[Document(id='36550579-0188-4a69-bb50-e2e148c6dd45', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.'),
 Document(id='d5c8c1ca-c61e-43c8-b1fc-bf4193c189d3', metadata={'source': 'langchain_sample.txt'}, page_content='Agents in LangChain are chains that use LLMs to decide which tools to use and in what order. This makes them suitable for multi-step tasks like question answering with search and code execution.'),
 Document(id='d2024728-1dbe-49ab-8236-4ee6aa9b2a4a', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components

In [23]:
chain=prompt| llm | StrOutputParser()
chain

PromptTemplate(input_variables=['documents', 'question'], input_types={}, partial_variables={}, template='\nYou are a helpful assistant. Your task is to rank the following documents from most to least relevant to the user\'s question.\n\nUser Question: "{question}"\n\nDocuments:\n{documents}\n\nInstructions:\n- Think about the relevance of each document to the user\'s question.\n- Return a list of document indices in ranked order, starting from the most relevant.\n\nOutput format: comma-separated document indices (e.g., 2,1,3,0,...)\n')
| Ollama(model='phi3')
| StrOutputParser()

In [24]:
doc_lines = [f"{i+1}. {doc.page_content}" for i, doc in enumerate(retrieved_docs)]
formatted_docs = "\n".join(doc_lines)

In [25]:
doc_lines

['1. LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.',
 '2. Agents in LangChain are chains that use LLMs to decide which tools to use and in what order. This makes them suitable for multi-step tasks like question answering with search and code execution.',
 '3. LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.',
 '4. LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.',
 '5. LangChain supports hybrid retrieval by combining BM25 and dense similarity score

In [26]:
formatted_docs

'1. LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.\n2. Agents in LangChain are chains that use LLMs to decide which tools to use and in what order. This makes them suitable for multi-step tasks like question answering with search and code execution.\n3. LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.\n4. LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.\n5. LangChain supports hybrid retrieval by combining BM25 and dense similarity scores. This appro

In [27]:
response = chain.invoke({"question": query, "documents": formatted_docs})
response

"3,1,4,8,5,7,2,0\n\nRationale for Ranking:\n3 The first document is the most relevant as it directly addresses LangChain's capabilities and tools that can be used to build an application capable of question answering based on documents. It provides a comprehensive overview pertinent to user needs.\n1 This next-to-first document also contains valuable information about agents within LangChain, which are essential components in building such applications as it mentions their role in multi-step tasks like the one described by the user's question. However, since this is less detailed than Document 3 regarding specific functionalities and more focused on workflow logic rather than direct application creation guidance, it takes a secondary place.\n4 Understanding LangChain’s support for tool integration provides context about how external information can supplement LLM capabilities which enhances the ability to answer user-generated questions accurately based on documents but does not direct

In [28]:
# Step 5: Parse and rerank
indices = [int(x.strip()) - 1 for x in response.split(",") if x.strip().isdigit()]
indices

[2, 0, 3, 7, 4, 6, 1]

In [29]:
reranked_docs = [retrieved_docs[i] for i in indices if 0 <= i < len(retrieved_docs)]
reranked_docs

[Document(id='d2024728-1dbe-49ab-8236-4ee6aa9b2a4a', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.'),
 Document(id='36550579-0188-4a69-bb50-e2e148c6dd45', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.'),
 Document(id='1eb95e3c-7bc5-4bd2-96c0-c614f49069c6', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond mo

In [30]:
# final 
prompt = PromptTemplate.from_template("""  
You are a helpful assistant. Your task is to answer the user's question based on the provided context.
"context":{context}
"qustion":{question}
     """)


In [32]:
from typing import List
# Format documents for the prompt
def format_docs(docs: List[Document]) -> str:
    """Format documents for insertion into prompt"""
    formatted = []
    for i, doc in enumerate(docs):
        source = doc.metadata.get('source', 'Unknown')
        formatted.append(f"Document {i+1} (Source: {source}):\n{doc.page_content}")
    return "\n\n".join(formatted)

In [35]:
# Create lcel
chain = (
    {"context": ensemble_retriever|format_docs, "question":RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()

)

In [37]:
query = "How can I use Langchain to build an application that can answer questions based on a document?"

res = chain.invoke(query)
res

'LangChain provides tools for building multi-step applications where LLMs decide which tool to use next in the process of answering queries like your question - ["Can you help me with developing my project?"]. It integrates several third-party services and includes components such as prompt management, chains (the agents), memory, and retrieval methods.\n\nTo build a document-based question-answering application using LangChain: \n1) Import the required libraries like langchain_openai or any other service provider you want to use in conjunction with OpenAI\'s GPT models for generating responses. For example:\n```python\nfrom langchain import OpenAIInteractionAgent, ChainBase, PromptTemplate\n\n# Define a custom prompt using LangChain template syntax and the LLM agent name "openai" \ntemplate = """Given my document {{document_contents}}, please answer this question in detail. The context for your response should be clear and concise: \'{{question}}\'."""\nagent = OpenAIInteractionAgent(